# Task 1: Web Scraping
### Evelyn Valeria Sarmiento Vásquez

In [ ]:
#Instalando todas las librerías necesarias: 
#!python -m pip install selenium webdriver-manager lxml tqdm unidecode



   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.6 MB 6.3 MB/s eta 0:00:02
   ------------ --------------------------- 2.9/9.6 MB 8.8 MB/s eta 0:00:01
   -------------------- ------------------- 5.0/9.6 MB 9.6 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.6 MB 9.6 MB/s eta 0:00:01
   ------------------------------------ --- 8.7/9.6 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 8.9 MB/s  0:00:01

   ---------------------------------------- 0/9 [wsproto]
  Attempting uninstall: urllib3
   ---------------------------------------- 0/9 [wsproto]
    Found existing installation: urllib3 2.5.0
   ---------------------------------------- 0/9 [wsproto]
    Uninstalling urllib3-2.5.0:
   ---------------------------------------- 0/9 [wsproto]
      Successfully uninstalled urllib3-2.5.0
   ---------------------------------------- 0/9 [wsproto]
   ---- -----------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [24]:
#Importando las librerías que se van a necesitar:

import base64
import time
import os

import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager



In [28]:
#Abriendo la url de la UNMSM:

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)
driver.maximize_window()

url = 'https://admision.unmsm.edu.pe/Website20262/A/A.html'
driver.get(url)



In [29]:

#Creación de funciones auxiliares:

#Para decodificar información oculta (en caso existiese)

def decode_obfuscated(data_auth):
    try:
        return base64.b64decode(data_auth).decode("utf-8")
    except Exception:
        return data_auth

#Para en vez de almacenar 50 observaciones por página, eliminar el límite de observaciones por página y mostrar todas a la vez

def show_all_rows(driver):
    driver.execute_script(
        "try { $('#tablaPostulantes').DataTable().page.len(-1).draw(); } catch(e) {}"
    )
    time.sleep(2)


#Para extraer todas las observaciones de una carrera:

def extract_all_rows(driver):
    show_all_rows(driver)
    rows = []
    tbody = driver.find_element(By.CSS_SELECTOR, "#tablaPostulantes tbody")
    for tr in tbody.find_elements(By.TAG_NAME, "tr"):
        tds = tr.find_elements(By.TAG_NAME, "td")
        row = []
        for td in tds:
            obf = td.find_elements(By.CLASS_NAME, "obfuscated")
            if obf:
                row.append(decode_obfuscated(obf[0].get_attribute("data-auth")))
            else:
                row.append(td.text.strip())
        if row:
            rows.append(row)
    return rows


#Extrae todas las filas de la carrera y agrega el nombre de la carrera como una columna adicional (ya que vamos a juntar las observaciones de todas las carreras y necesitamos una forma de cómo diferenciarlas)

def scrape_career(driver, url, career_name):
    driver.get(url)
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.ID, "tablaPostulantes"))
    )
    time.sleep(1)
    rows = extract_all_rows(driver)
    return [[career_name] + row for row in rows]


In [30]:

#Extraer todas las carreras 

INDEX_URL = "https://admision.unmsm.edu.pe/Website20262/A/A.html"

driver.get(INDEX_URL)
WebDriverWait(driver, 15).until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a[href*='results.html']"))
)

links = driver.find_elements(By.CSS_SELECTOR, "a[href*='results.html']")
careers = [
    (link.text.strip(), link.get_attribute("href"))
    for link in links
    if link.text.strip()
]

print(f"Total de carreras encontradas: {len(careers)}")
print("Ejemplo de las 5 primeras carreras:")
for name, url in tqdm(careers[:5], desc="Verificando links"):
    print(f"  {name}  →  {url}")


Total de carreras encontradas: 111
Ejemplo de las 5 primeras carreras:


Verificando links: 100%|██████████| 5/5 [00:00<00:00, 4233.25it/s]

  ADMINISTRACIÓN  →  https://admision.unmsm.edu.pe/Website20262/A/091/results.html
  ADMINISTRACIÓN - CHILCA  →  https://admision.unmsm.edu.pe/Website20262/A/0914/results.html
  ADMINISTRACIÓN - HUARAL  →  https://admision.unmsm.edu.pe/Website20262/A/0912/results.html
  ADMINISTRACIÓN - S.J.L  →  https://admision.unmsm.edu.pe/Website20262/A/0911/results.html
  ADMINISTRACIÓN - VILLA RICA  →  https://admision.unmsm.edu.pe/Website20262/A/0915/results.html
